In [3]:
import torch
import torch.nn as nn 
import random, math
import torch.nn.functional as F 
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageDraw
import matplotlib

In [4]:
IMAGE_SIZE=32
EMDED_DIM=64
ATTENTION_HEAD=4
BATCH_SIZE=12
EPOCHS=10
LR=3e-10
TEMP=0.07

In [6]:
device=torch.device('mps' if torch.mps.is_available() else 'cpu')

In [9]:

colors = ['red', 'green', 'blue', 'yellow', 'purple', 'orange', 'pink', 'brown', 'gray']
shapes = ['square', 'circle', 'triangle']
positions = ['left', 'center', 'right', 'top', 'bottom', 'top-left', 'top-right', 'bottom-left', 'bottom-right']


In [15]:

from PIL import Image, ImageDraw

IMG_SIZE = 64

def draw_sample(color, shape, position, img_size=IMG_SIZE):
    img = Image.new('RGB', (img_size, img_size), 'white')
    draw = ImageDraw.Draw(img)

    margin = 6
    h = w = img_size - 2 * margin

    # Calculate x coordinates
    if 'left' in position:
        x0 = margin
        x1 = margin + w // 2
    elif 'top-left' in position:
        x0 = margin
        x1 = margin + w // 2
    elif 'bottom-left' in position:
        x0 = margin
        x1 = margin + w // 2
    elif 'right' in position:
        x0 = margin + w // 2
        x1 = img_size - margin
    elif 'top-right' in position:
        x0 = margin + w // 2
        x1 = img_size - margin
    elif 'bottom-right' in position:
        x0 = margin + w // 2
        x1 = img_size - margin
    else:  # center or vertical positions
        x0 = margin + w // 4
        x1 = margin + 3 * w // 4

    # Calculate y coordinates
    if 'top' in position:
        y0 = margin
        y1 = margin + h // 2
    elif 'top-left' in position:
        y0 = margin
        y1 = margin + h // 2
    elif 'top-right' in position:
        y0 = margin
        y1 = margin + h // 2
    elif 'bottom' in position:
        y0 = margin + h // 2
        y1 = img_size - margin
    elif 'bottom-left' in position:
        y0 = margin + h // 2
        y1 = img_size - margin
    elif 'bottom-right' in position:
        y0 = margin + h // 2
        y1 = img_size - margin
    else:  # center or horizontal positions
        y0 = margin + h // 4
        y1 = margin + 3 * h // 4

    if shape == 'square':
        draw.rectangle([x0, y0, x1, y1],
                       fill=color, outline='black')
    elif shape == 'circle':
        draw.ellipse([x0, y0, x1, y1],
                     fill=color, outline='black')
    else:  # triangle
        draw.polygon(
            [(x0 + (x1 - x0) // 2, y0),
             (x0, y1),
             (x1, y1)],
            fill=color,
            outline='black'
        )

    return img


In [ ]:
#building the dataset 
import numpy as np

class shapedataset():
    def __init__(self):
        self.images=[]
        self.captions=[]

        for c in colors:
            for s in shapes:
                for p in positions:
                    img=draw_sample(c,s,p)
                    cap=f"{c} {s} {p}"

                    self.images.append(torch.from_numpy(np.array(img, copy=True)).permute(2, 0, 1).float() / 255.0)
                    self.captions.append(cap)

        self.vocab, self.word2idx=self.build_vocab(self.captions)

    def build_vocab(self,texts):
        words= sorted({w for t in texts for w in t.split()})
        vocab= ['[CLS]']+ words
        w2i={w:i for i,w in enumerate(vocab)}
        return vocab,w2i
    def encode_text(self,text):
        toks = [self.word2idx['[CLS]']] + [self.word2idx[w] for w in text.split()]
        return torch.tensor(toks, dtype=torch.long)
    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
         return self.images[i], self.encode_text(self.captions[i]), self.captions[i]


full_ds = shapedataset()
VOCAB_SIZE = len(full_ds.vocab)

/var/folders/k3/7xcqk4_175xf65lxppc0gldh0000gn/T/ipykernel_2674/754314792.py:15: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  self.images.append(torch.from_numpy(np.asarray(img)).permute(2, 0, 1).float() / 255.0)
